# Demo A &mdash; LLM hypothesis &rarr; TriageSpec (no RAG)

**One-shot, no human-in-the-loop, no retry.** A natural-language goal goes to the LLM,
which proposes a `Hypothesis` (constraints / ranking targets / element rules) via Bedrock
structured output; `compile_spec` then deterministically assembles the proposals into a
frozen `TriageSpec` that the deterministic pipeline would consume.

This is a *thin* demo: the prompt is written inline (the real prompt module is #22), and
there is **no validation retry** &mdash; the LLM occasionally emits output the `Hypothesis`
schema rejects (~15% of calls, measured), so a run may simply fail. That is expected.

**Needs AWS credentials** for Bedrock (env vars, a profile, or `~/.aws/credentials`).

In [ ]:
from dotenv import load_dotenv

from materials_triage.agent.llm import HypothesisProvider
from materials_triage.core.hypothesis import compile_spec

load_dotenv()  # surface .env creds; ~/.aws/credentials also works (botocore auto-detects)

In [ ]:
GOAL = (
    "Find promising oxide dielectric candidates for thin-film experiments. "
    "Prefer thermodynamically stable materials, wide band gaps, non-toxic "
    "elements, simple compositions, and public evidence. "
    "Return a ranked shortlist with caveats."
)

PROMPT = f"""You are a materials-science research assistant. Convert the scientist's goal
below into a triage specification that bridges the fuzzy goal to concrete, queryable
Materials Project properties.

Emit a Hypothesis:
- proposals: each is a constraint (hard min/max filter on one property), a ranking_target
  (a property, a direction "minimize"/"maximize", and a weight as a proportional share),
  or an element_rule (require/exclude element symbols).
- mechanism: a short mechanistic "why".

Use ONLY these property names (units):
  band_gap (eV), energy_above_hull (eV/atom), formation_energy_per_atom (eV/atom),
  density (g/cm^3), bulk_modulus (GPa), shear_modulus (GPa).

Rules:
- Include AT LEAST ONE hard constraint.
- Give each proposal a brief rationale and a confidence in (0, 1].
- No literature was retrieved for this demo, so leave citations empty.

GOAL: {GOAL}
"""
print(PROMPT)

In [5]:
provider = HypothesisProvider()  # real Bedrock transport (built lazily)
hypothesis = provider.propose(PROMPT)  # ONE shot - may raise if the LLM output is malformed

print("MECHANISM:\n", hypothesis.mechanism, "\n")
print(len(hypothesis.proposals), "proposals:")
for p in hypothesis.proposals:
    payload = getattr(p, p.kind)
    print(f"  - [{p.kind}] {payload}  (conf={p.confidence})")
    print(f"      rationale: {p.rationale}")

MECHANISM:
 Thin-film dielectrics require a tight balance: thermodynamic stability ensures the material survives processing (hard constraint on energy_above_hull); wide band gaps (hard constraint + ranking) minimize leakage and maximize breakdown voltage. Secondary ranking by formation energy, density, and element safety biases the shortlist toward synthetically accessible, chemically stable, non-toxic oxides. The composition-scoping rules (require O, exclude toxic metals) narrow the search to commercially and environmentally viable candidates suitable for device integration. 

7 proposals:
  - [constraint] property_name='energy_above_hull' min=None max=0.05  (conf=0.95)
      rationale: Thermodynamic stability is critical for thin-film processing. Materials far above the convex hull are metastable and prone to decomposition during deposition or annealing. A tight constraint (≤0.05 eV/atom) ensures we prioritize kinetically accessible, stable phases.
  - [constraint] property_name='ban

In [6]:
spec = compile_spec(hypothesis.proposals)  # deterministic; raises if e.g. < 1 constraint

print("TriageSpec")
print("  constraints:")
for c in spec.constraints:
    print(f"    {c.property_name}: min={c.min} max={c.max}")
print("  ranking_targets:")
for t in spec.ranking_targets:
    print(f"    {t.property_name}: {t.direction} weight={t.weight:.3f}")
print("  required_elements:", sorted(spec.required_elements))
print("  excluded_elements:", sorted(spec.excluded_elements))

TriageSpec
  constraints:
    energy_above_hull: min=None max=0.05
    band_gap: min=4.0 max=None
  ranking_targets:
    band_gap: maximize weight=0.350
    formation_energy_per_atom: minimize weight=0.350
    density: maximize weight=0.300
  required_elements: ['O']
  excluded_elements: ['As', 'Cd', 'Cr', 'Hg', 'Pb']


**What just happened:** the LLM did the *interpretive* bridge (goal &rarr; proxy
properties); `compile_spec` normalized the ranking weights to sum to 1 and assembled a
frozen `TriageSpec`. From here the deterministic pipeline (`retrieve &rarr;
apply_hard_filters &rarr; rank`) would run &mdash; every number sourced from Materials
Project, never the LLM.

If a cell raised, that is the one-shot reality: no retry loop yet (deferred to the
orchestrator, #23). Re-run to try again.